# Logit Lens — exploración inicial (2026-08-17)

**Objetivo de este notebook:** entender de forma práctica qué es el *logit lens* antes de aplicarlo como parte del bloque A (feature study) del capítulo de ViT — ver `CLAUDE.md`, punto **A.2**: *"proyectar el CLS token intermedio de cada bloque (post-LN) a través del clasificador final para rastrear en qué capa 'se decide' la clase"*.

No es todavía el análisis comparativo (baseline vs. penalizado) — es la prueba de concepto: cargar un modelo ya entrenado (uno de los checkpoints que ya existen en `experimentation/experiments/`) y ver, capa a capa, cómo evoluciona la predicción de clase.

## ¿Qué es el logit lens?

La idea original es de **nostalgebraist (2020)**, *"interpreting GPT: the logit lens"* (post en LessWrong: [lesswrong.com/posts/AcKRB8wDpdaN6v6ru](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)): en un modelo tipo GPT, cada bloque transformer va sumando (vía conexiones residuales) al *residual stream*. La observación es que el embedding de salida (la matriz que convierte el estado oculto final en logits sobre el vocabulario) se puede aplicar **también a los estados intermedios**, no solo al final — como poner una "lupa" (lens) en cada capa para leer, en el espacio de logits, qué predeciría el modelo si se detuviera ahí. Esto revela que muchos modelos van "decidiéndose" gradualmente, capa a capa, en vez de decidir todo de golpe en la última capa.

Para un ViT de clasificación (no autoregresivo, sin vocabulario) el equivalente es: en vez de proyectar cada token contra el vocabulario, se proyecta el **token [CLS]** de cada bloque contra el **clasificador final** (la capa lineal que da los logits de clase). Es exactamente lo que hace este notebook. Esta adaptación está explícitamente estudiada en:

- Vilas, Schaumlöffel, Roig (NeurIPS 2023), *"Analyzing Vision Transformers for Image Classification in Class Embedding Space"* — [arxiv.org/abs/2310.18969](https://arxiv.org/abs/2310.18969) · [código](https://github.com/martinagvilas/vit-cls_emb). Proyectan representaciones intermedias de un ViT al espacio de embeddings de clase (misma idea que el logit lens, aplicada a visión) y muestran que la identificabilidad de la clase correcta crece progresivamente capa a capa, no solo en el token [CLS] sino también en los tokens de parche.
- Belrose et al. (2023), *"Eliciting Latent Predictions from Transformers with the Tuned Lens"* — [arxiv.org/abs/2303.08112](https://arxiv.org/abs/2303.08112), la evolución del logit lens: en vez de reusar el clasificador final tal cual, entrena una pequeña transformación afín por capa para corregir el sesgo del logit lens "crudo". Buena referencia para cuando este notebook exploratorio se convierta en el análisis real del capítulo.
- Adjileye (Medium), *"Unlocking Visual Insights: Applying the 'Logit Lens' to Image Data with Vision Transformers"* (ya citado en `CLAUDE.md`) — versión aplicada y visual de lo mismo.

**Por qué es barato en esta arquitectura:** en `ViTForClassfication` (`modules/model.py`) el clasificador es una única capa lineal aplicada directamente al CLS del último bloque (`self.classifier(encoder_output[:, 0, :])`), sin LayerNorm final adicional. Eso significa que el logit lens es literalmente aplicar esa misma capa lineal al CLS de *cualquier* bloque intermedio — no hay que entrenar nada nuevo ni reimplementar el clasificador.

## Qué hace este notebook

1. Carga un checkpoint ya entrenado (`vit-with-15-epochs-CIFAR10`, baseline sin penalización, ~60% accuracy — el más entrenado de los baseline disponibles, elegido porque queremos ver una progresión con señal real, no ruido de un modelo sin entrenar).
2. Implementa `logit_lens(modelo, x)`: para cada bloque (incluyendo el embedding como "capa 0"), toma el CLS y lo pasa por el clasificador final.
3. Verifica que la capa final del logit lens coincide exactamente con el forward normal del modelo (sanity check).
4. Mide accuracy por capa sobre un lote de imágenes de test — el gráfico central: ¿en qué bloque empieza a acertar la clase?
5. Mira ejemplos individuales: cómo evoluciona la probabilidad asignada a la clase correcta, capa a capa, y qué predice cada capa en concreto.

Cuando esto tenga sentido, el siguiente paso natural es repetir exactamente este notebook cargando el checkpoint **interpretable** (`vit-with-10-epochs-interpretable-CIFAR10`) y comparar en qué capa "se decide" cada variante — eso es ya el análisis real de A.2.

In [ ]:
import json
import sys
from pathlib import Path

# Igual que en notebooks/09082026_effective_rank/effective_rank.ipynb:
# esta notebook vive en notebooks/<algo>/, dos niveles debajo de la raíz del proyecto.
project_root = Path.cwd().resolve().parents[1]
print(f"Project root: {project_root}")
sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision

from modules.model import ViTForClassfication
from modules.datasets import CIFAR10Dataset

torch.manual_seed(0)

**Todas las gráficas de este notebook se guardan solas** en esta misma carpeta, como `<nombre_de_la_grafica>_<nombre_del_modelo>.png` (sin que tengas que hacer nada extra). Con el `EXPERIMENT_NAME` que dejaste activo en la sección 1, vas a obtener:

- `accuracy_por_capa_<modelo>.png` (sección 4)
- `probabilidad_por_ejemplo_<modelo>.png` (sección 5)
- `predicciones_por_capa_<modelo>.png` (sección 5)
- `matriz_confusion_<modelo>.png` — esta la genera dos veces sola, una por cada checkpoint (baseline e interpretable), sin importar cuál `EXPERIMENT_NAME` hayas dejado activo arriba (sección 6)

Si quieres las cuatro primeras también para el otro checkpoint, cambia `EXPERIMENT_NAME` en la sección 1 y vuelve a correr el notebook — no se pisan entre sí porque el nombre del modelo va en el archivo.

## 1. Cargar un modelo ya entrenado

En vez de un modelo dummy sin entrenar (con pesos aleatorios el logit lens solo mostraría ruido, nada interesante que interpretar), usamos un checkpoint real que ya existe en `experimentation/experiments/vit-with-15-epochs-CIFAR10/` — el baseline (sin penalización de similitud entre cabezas) más entrenado que hay, ~60% de accuracy en CIFAR-10.

Cargamos con `map_location="cpu"` explícito (igual que `effective_rank.ipynb`) para que corra igual sin importar si hay GPU/MPS disponible — el checkpoint se guardó originalmente desde un dispositivo MPS.

In [ ]:
# Los dos checkpoints disponibles (mismo config, misma seed de datos, ver CLAUDE.md):
#   "vit-with-15-epochs-CIFAR10"              -- baseline, SIN penalizacion de similitud entre cabezas
#   "vit-with-10-epochs-interpretable-CIFAR10" -- CON penalizacion (el "interpretable")
# Cambia esta linea y vuelve a correr desde aqui para explorar el otro modelo
# en las secciones 1-5. La seccion 6 (matriz de confusion) ya corre las DOS
# automaticamente, sin que tengas que tocar esto.
EXPERIMENT_NAME = "vit-with-15-epochs-CIFAR10"
exp_dir = project_root / "experimentation" / "experiments" / EXPERIMENT_NAME

with open(exp_dir / "config.json") as f:
    config = json.load(f)
with open(exp_dir / "metrics.json") as f:
    metrics = json.load(f)

model = ViTForClassfication(config)
state_dict = torch.load(exp_dir / "model_final.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

print(f"Checkpoint: {EXPERIMENT_NAME}  (accuracy de entrenamiento: {metrics['accuracy']:.4f})")
print(f"Config: {config}")
print(f"Bloques transformer: {len(model.encoder.blocks)}")

## 2. Datos de test (CIFAR-10)

Apuntamos `root` directamente a `experimentation/data`, donde ya está descargado CIFAR-10 (evita volver a descargar ~170MB por usar la carpeta relativa `./data` de `CIFAR10Dataset.get_loaders`).

In [ ]:
dataset = CIFAR10Dataset()
dataset.testset = torchvision.datasets.CIFAR10(
    root=str(project_root / "experimentation" / "data"),
    train=False,
    download=True,
    transform=dataset.test_transform,
)
testset = dataset.testset
classes = dataset.classes
print(f"Test set: {len(testset)} imágenes, {len(classes)} clases: {classes}")

## 3. La función `logit_lens`

Corre el forward manualmente bloque por bloque (en vez de llamar `model(x)` de una), guardando el CLS token después de cada paso, y proyecta cada uno con el clasificador final. La "capa 0" es el CLS justo después de los embeddings (antes de cualquier bloque transformer) — sirve de referencia de qué tan bien predice el clasificador si no hubiera pasado por ningún bloque (debería estar cerca del azar, 10%).

In [ ]:
@torch.no_grad()
def logit_lens(model, x):
    """
    x: (B, C, H, W)
    Devuelve (layer_names, all_probs): all_probs es una lista de longitud
    num_hidden_layers + 1 con las probabilidades softmax (B, num_classes) de
    proyectar el CLS token a través del clasificador final, capturado después
    de los embeddings (capa 0) y después de cada bloque.
    """
    embedding_output = model.embedding(x)
    cls_states = [embedding_output[:, 0, :]]

    h = embedding_output
    for block in model.encoder.blocks:
        h, _ = block(h, output_attentions=False)
        cls_states.append(h[:, 0, :])

    layer_names = ["embeddings"] + [f"block {i+1}" for i in range(len(model.encoder.blocks))]
    all_probs = [F.softmax(model.classifier(cls), dim=-1) for cls in cls_states]
    return layer_names, all_probs

In [ ]:
# Sanity check: la última capa del logit lens debe coincidir EXACTAMENTE
# con el forward normal del modelo (es la misma cuenta, solo que la hacemos
# a mano capa por capa en vez de dejar que `model.forward` la haga de una).
sample_images, sample_labels = dataset.get_samples_from_indices(range(8), device="cpu", set="test")

layer_names, probs_per_layer = logit_lens(model, sample_images)

with torch.no_grad():
    logits, _ = model(sample_images, output_attentions=False)
    direct_probs = F.softmax(logits, dim=-1)

max_diff = (direct_probs - probs_per_layer[-1]).abs().max().item()
print(f"Diferencia máxima entre logit-lens (última capa) y forward normal: {max_diff:.2e}")
assert max_diff < 1e-6, "La última capa del logit lens debería ser idéntica al forward normal del modelo"
print("OK — el logit lens está bien implementado: en la última capa reproduce exactamente la predicción real del modelo.")

## 4. ¿En qué capa "se decide" la clase? — accuracy por capa

Esta es la pregunta central de A.2. Tomamos un lote de imágenes de test, corremos `logit_lens`, y en cada capa medimos qué fracción de las predicciones (top-1 del logit lens en esa capa) coincide con la etiqueta real. Si la curva sube rápido y se aplana antes del último bloque, quiere decir que el modelo "ya sabe" la clase antes de terminar — esa es la señal que se compara después entre baseline y penalizado.

In [ ]:
N_EVAL = 500  # tamaño del lote de evaluación (CPU-friendly)

eval_images, eval_labels = dataset.get_samples_from_indices(range(N_EVAL), device="cpu", set="test")
layer_names, probs_per_layer = logit_lens(model, eval_images)

layer_accuracies = []
for probs in probs_per_layer:
    preds = probs.argmax(dim=-1)
    acc = (preds == eval_labels).float().mean().item()
    layer_accuracies.append(acc)

for name, acc in zip(layer_names, layer_accuracies):
    print(f"{name:>10}: accuracy = {acc:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(layer_names, layer_accuracies, marker="o", linewidth=2)
ax.axhline(1 / len(classes), color="gray", linestyle="--", linewidth=1, label="azar (1/10)")
ax.axhline(metrics["accuracy"], color="green", linestyle=":", linewidth=1, label="accuracy final reportada en metrics.json")
ax.set_ylabel("accuracy (logit lens, top-1)")
ax.set_xlabel("capa")
ax.set_title(f"Logit lens — accuracy por capa ({EXPERIMENT_NAME}, n={N_EVAL})")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()

output_path = Path.cwd() / f"accuracy_por_capa_{EXPERIMENT_NAME}.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {output_path}")

## 5. Mirando ejemplos individuales

Para un puñado de imágenes: la probabilidad que el logit lens le asigna a la **clase verdadera** en cada capa (el gráfico clásico de logit lens), y qué clase predice top-1 en cada capa. Una imagen "fácil" debería subir rápido y quedarse arriba; una imagen mal clasificada al final puede mostrar dudas o cambios de opinión entre capas.

In [ ]:
N_EXAMPLES = 8
ex_images, ex_labels = dataset.get_samples_from_indices(range(N_EXAMPLES), device="cpu", set="test")
layer_names, probs_per_layer = logit_lens(model, ex_images)

# (n_layers, n_examples) -- probabilidad asignada a la clase verdadera en cada capa
true_class_probs = torch.stack([
    probs[torch.arange(N_EXAMPLES), ex_labels] for probs in probs_per_layer
])  # (n_layers, N_EXAMPLES)

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(N_EXAMPLES):
    final_pred_ok = (probs_per_layer[-1][i].argmax().item() == ex_labels[i].item())
    style = "-" if final_pred_ok else "--"
    ax.plot(
        layer_names,
        true_class_probs[:, i].tolist(),
        marker="o",
        linestyle=style,
        label=f"{classes[ex_labels[i]]} ({'ok' if final_pred_ok else 'mal clasificada'})",
    )
ax.set_ylabel("P(clase verdadera)")
ax.set_xlabel("capa")
ax.set_title("Logit lens por ejemplo — probabilidad de la clase verdadera")
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()

output_path = Path.cwd() / f"probabilidad_por_ejemplo_{EXPERIMENT_NAME}.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {output_path}")

In [ ]:
# Grilla de imágenes con la secuencia de predicciones top-1 por capa, para verlo
# de forma muy concreta: "esto es lo que el modelo hubiera dicho si se detiene aquí".
images_unnorm = ex_images * 0.5 + 0.5  # deshacer Normalize((0.5,...), (0.5,...))

ncols = 4
nrows = (N_EXAMPLES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3.2))
axes = axes.flatten()

for i in range(N_EXAMPLES):
    top1_per_layer = [classes[probs_per_layer[l][i].argmax().item()] for l in range(len(layer_names))]
    sequence = " → ".join(top1_per_layer)
    true_label = classes[ex_labels[i]]
    axes[i].imshow(images_unnorm[i].permute(1, 2, 0).numpy())
    axes[i].set_title(f"real: {true_label}\n{sequence}", fontsize=7)
    axes[i].axis("off")

for ax in axes[N_EXAMPLES:]:
    ax.axis("off")

plt.tight_layout()

output_path = Path.cwd() / f"predicciones_por_capa_{EXPERIMENT_NAME}.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {output_path}")

## 6. Matriz de confusión por capa (todo el set de evaluación, baseline y interpretable)

Para cada checkpoint (baseline y penalizado), y para cada capa (embeddings + cada bloque), la matriz de confusión de lo que predice el logit lens en esa capa — sobre **todo** el set de evaluación de CIFAR-10 (10 000 imágenes, no solo la muestra de `N_EVAL` que usan las secciones 4 y 5). Las matrices de un mismo checkpoint se combinan en una sola figura (una fila, una columna por capa) y se guardan como imagen **en esta misma carpeta**, con el nombre del checkpoint (sin extensión) — por ejemplo `vit-with-15-epochs-CIFAR10.png`.

No hace falta volver a cargar nada a mano: esta sección carga los dos checkpoints por su cuenta (usa una copia local del modelo, no toca la variable `model` de la sección 1) y genera los dos archivos en una sola corrida.

In [ ]:
from pathlib import Path

CHECKPOINTS_FOR_CONFUSION = [
    "vit-with-15-epochs-CIFAR10",               # baseline
    "vit-with-10-epochs-interpretable-CIFAR10", # interpretable (penalizado)
]
CM_BATCH_SIZE = 500  # tamano de lote para no cargar las 10000 imagenes de una


def load_checkpoint(experiment_name):
    exp_dir = project_root / "experimentation" / "experiments" / experiment_name
    with open(exp_dir / "config.json") as f:
        cfg = json.load(f)
    m = ViTForClassfication(cfg)
    m.load_state_dict(torch.load(exp_dir / "model_final.pt", map_location="cpu"))
    m.eval()
    return m, cfg


def confusion_matrix_np(true_labels, pred_labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(cm, (true_labels, pred_labels), 1)
    return cm


def logit_lens_predictions_full(m, batch_size=CM_BATCH_SIZE):
    """Corre logit_lens sobre TODO el testset, en lotes, y devuelve
    (layer_names, true_labels_np, [preds_por_capa_np, ...])."""
    n_total = len(testset)
    all_true, preds_per_layer = [], None
    for start in range(0, n_total, batch_size):
        idx = range(start, min(start + batch_size, n_total))
        imgs, labels = dataset.get_samples_from_indices(idx, device="cpu", set="test")
        names, probs = logit_lens(m, imgs)
        if preds_per_layer is None:
            preds_per_layer = [[] for _ in probs]
        for layer_idx, p in enumerate(probs):
            preds_per_layer[layer_idx].append(p.argmax(dim=-1))
        all_true.append(labels)
        print(f"  {min(start + batch_size, n_total)}/{n_total}", end="\r")
    print()
    true_labels_np = torch.cat(all_true).numpy()
    preds_per_layer_np = [torch.cat(p).numpy() for p in preds_per_layer]
    return names, true_labels_np, preds_per_layer_np


import numpy as np

for cm_experiment_name in CHECKPOINTS_FOR_CONFUSION:
    print(f"\n=== {cm_experiment_name} ===")
    cm_model, cm_config = load_checkpoint(cm_experiment_name)
    cm_layer_names, true_labels_np, preds_per_layer_np = logit_lens_predictions_full(cm_model)

    n_layers = len(cm_layer_names)
    fig, axes = plt.subplots(1, n_layers, figsize=(3.6 * n_layers, 4.2))
    for ax, layer_name, preds in zip(axes, cm_layer_names, preds_per_layer_np):
        cm = confusion_matrix_np(true_labels_np, preds, len(classes))
        acc = (true_labels_np == preds).mean()
        ax.imshow(cm, cmap="Blues")
        for i in range(len(classes)):
            for j in range(len(classes)):
                if cm[i, j] > 0:
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6,
                            color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=90, fontsize=7)
        ax.set_yticks(range(len(classes)))
        ax.set_yticklabels(classes, fontsize=7)
        ax.set_xlabel("predicha", fontsize=8)
        ax.set_title(f"{layer_name}\nacc={acc:.3f}", fontsize=10)
    axes[0].set_ylabel("real", fontsize=8)

    fig.suptitle(f"Matriz de confusion por capa -- {cm_experiment_name} (n={len(testset)})", y=1.04, fontsize=12)
    plt.tight_layout()

    output_path = Path.cwd() / f"matriz_confusion_{cm_experiment_name}.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Guardado: {output_path}")

**Qué mirar en las imágenes guardadas:** dentro de cada matriz, la diagonal es lo que el modelo acierta en esa capa — si ya está oscura/marcada en `block 2` o `block 3`, la clase se decide temprano. Fuera de la diagonal, los pares de clases que más se confunden entre sí (por ejemplo gato/perro, o los distintos vehículos) — y si esa confusión aparece desde una capa temprana o solo se resuelve al final. Comparar el archivo `vit-with-15-epochs-CIFAR10.png` contra `vit-with-10-epochs-interpretable-CIFAR10.png` lado a lado es la comparación real que pide A.2 de `CLAUDE.md`.

## Para reflexionar / siguientes pasos

- ¿En qué capa deja de subir la accuracy agregada? ¿Es ya el bloque 2-3 de 4, o solo se decide en el último?
- ¿Las imágenes mal clasificadas al final ya se veían "dudosas" desde capas tempranas, o el modelo cambia de opinión tarde?
- Con un ViT de solo 4 bloques la ventana para ver progresión es corta — vale la pena repetir esto con más `N_EVAL` / más ejemplos individuales antes de sacar conclusiones.
- Las matrices de confusión de la sección 6 (baseline vs. interpretable, guardadas como imagen en esta carpeta) son la comparación real de A.2 — mira ahí si la penalización cambia en qué capa se decide la clase, y si cambia qué pares de clases se confunden entre sí.